In [7]:
import os
from IPython.display import clear_output
import time
import pandas as pd
import pickle
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable

In [5]:
# dylan instance details 
NEO4J_URI='neo4j+s://5fe6b298.databases.neo4j.io'
NEO4J_USERNAME='neo4j'
NEO4J_PASSWORD='V2S2unBr-ex60KUDHFzMH_KLkiPggGUR9UkEZYreAPM'
AURA_INSTANCEID='5fe6b298'
AURA_INSTANCENAME='Instance01'


driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [11]:
def listdir(directory:str) -> list:
    return [x for x in os.listdir(directory) if x[0] != '.']

data_directory = os.path.join(os.getcwd(),'data')
for season_directory in listdir(data_directory):
    print(season_directory)
    
    fixtures_df = pd.read_csv(os.path.join(data_directory, season_directory, f'fixtures_{season_directory}.csv'))
    match_reports = list()
    team_dict = dict()

    with open(os.path.join(data_directory, season_directory, f'match_report_{season_directory}.pkl'), "rb") as f:
        match_reports = pickle.load(f)
    with open(os.path.join(data_directory, season_directory, f'teams{season_directory}.pkl'), "rb") as f:
        team_dict = pickle.load(f)
    with open(os.path.join(data_directory, season_directory, f'players{season_directory}.pkl'), "rb") as f:
        player_details = pickle.load(f)

    #clean up
    fixtures_df = fixtures_df[fixtures_df['Wk'].notna()].reset_index(drop=True)
    fixtures_df[['Home_Score', 'Away_Score']] = fixtures_df['Score'].str.split('–', expand=True)
    fixtures_df['Wk'] = "GW" + fixtures_df['Wk'].astype(int).astype(str)
    fixtures_df = fixtures_df.rename(columns={'Wk':'GW'})

    print('the length of fixtures df is '+(str(len(fixtures_df))))

    for team_name, players_names_list in team_dict.items():
        for player in players_names_list:
            if player in player_details:
                player_details[player]['Team'] = team_name
    for player in player_details.keys():
        player_details[player]['Name'] = player
    player_details = list(player_details.values())
    
    game_weeks_data = [{"gw_number": f"GW{gw}", "season_name": season_directory} for gw in range(1, 39)]

# Transform fixtures_df to a list of dictionaries for UNWIND
    match_data = []
    player_stats_data = []
    goalkeeper_stats_data = []
    
    for i, row in fixtures_df[:len(match_reports)].iterrows():
        match_entry = {
            "match_id": f"{row['GW']}_{row['Home']}_vs_{row['Away']}",
            "date": row["Date"],
            "time": row["Time"],
            "home_team": row["Home"],
            "away_team": row["Away"],
            "gameweek": row["GW"],
            "venue": row["Venue"],
            "has_match_report": row["Match Report"] == "Match Report",
            "home_score": row["Home_Score"] if row["Match Report"] == "Match Report" else None,
            "away_score": row["Away_Score"] if row["Match Report"] == "Match Report" else None,
            "referee": row["Referee"] if row["Match Report"] == "Match Report" else None,
            "home_expected": row["xG"] if "xG" in row and row["Match Report"] == "Match Report" else None,
            "away_expected": row["xG.1"] if "xG.1" in row and row["Match Report"] == "Match Report" else None,
        }
        match_data.append(match_entry)
    
    # Process fixtures and match reports
        match_id = f"{row['GW']}_{row['Home']}_vs_{row['Away']}"
        home_team_name = row['Home']
        away_team_name = row['Away']
        home_team_stats = match_reports[i][0][0][:-1]
        away_team_stats = match_reports[i][1][0][:-1]
        home_team_goalie_stats = match_reports[i][0][1]
        away_team_goalie_stats = match_reports[i][1][1]
    
        # Prepare player stats data
        for team_name, team_stats in [(home_team_name, home_team_stats), (away_team_name, away_team_stats)]:
            for _, player_stats in team_stats.iterrows():
                player_stats_data.append({
                    "player_name": player_stats[('Unnamed: 0_level_0', 'Player')],
                    "position": player_stats[('Unnamed: 3_level_0', 'Pos')],
                    "match_id": match_id,
                    "team_name": team_name,
                    "goals": player_stats[('Performance', 'Gls')],
                    "assists": player_stats[('Performance', 'Ast')],
                    "penalties": player_stats[('Performance', 'PK')],
                    "shots": player_stats[('Performance', 'Sh')],
                    "shots_on_target": player_stats[('Performance', 'SoT')],
                    "touches": player_stats[('Performance', 'Touches')],
                    "tackles": player_stats[('Performance', 'Tkl')],
                    "interceptions": player_stats[('Performance', 'Int')],
                    "blocks": player_stats[('Performance', 'Blocks')],
                    "xg": player_stats[('Expected', 'xG')],
                    "npxg": player_stats[('Expected', 'npxG')],
                    "xag": player_stats[('Expected', 'xAG')],
                    "sca": player_stats[('SCA', 'SCA')],
                    "gca": player_stats[('SCA', 'GCA')],
                    "passes_completed": player_stats[('Passes', 'Cmp')],
                    "passes_attempted": player_stats[('Passes', 'Att')],
                    "pass_accuracy": player_stats[('Passes', 'Cmp%')],
                    "progressive_passes": player_stats[('Passes', 'PrgP')],
                    "carries": player_stats[('Carries', 'Carries')],
                    "progressive_carries": player_stats[('Carries', 'PrgC')],
                    "season_name": season_directory,
                })
    
        # Prepare goalie stats data
        for team_name, goalie_stats in [(home_team_name, home_team_goalie_stats), (away_team_name, away_team_goalie_stats)]:
            for _, goalie_stat in goalie_stats.iterrows():
                goalkeeper_stats_data.append({
                    "goalie_name": goalie_stat[('Unnamed: 0_level_0', 'Player')],
                    "match_id": match_id,
                    "team_name": team_name,
                    "minutes_played": goalie_stat[('Unnamed: 3_level_0', 'Min')],
                    "sota": goalie_stat[('Shot Stopping', 'SoTA')],
                    "goals_allowed": goalie_stat[('Shot Stopping', 'GA')],
                    "saves": goalie_stat[('Shot Stopping', 'Saves')],
                    "save_percentage": goalie_stat[('Shot Stopping', 'Save%')],
                    "psxg": goalie_stat[('Shot Stopping', 'PSxG')],
                    "passes_completed": goalie_stat[('Launched', 'Cmp')],
                    "passes_attempted": goalie_stat[('Launched', 'Att')],
                    "pass_accuracy": goalie_stat[('Launched', 'Cmp%')],
                    "gk_passes_attempted": goalie_stat[('Passes', 'Att (GK)')],
                    "throws": goalie_stat[('Passes', 'Thr')],
                    "launch_percentage": goalie_stat[('Passes', 'Launch%')],
                    "launch_average_length": goalie_stat[('Passes', 'AvgLen')],
                    "goal_kicks_attempted": goalie_stat[('Goal Kicks', 'Att')],
                    "goal_kick_launch_percentage": goalie_stat[('Goal Kicks', 'Launch%')],
                    "goal_kick_average_length": goalie_stat[('Goal Kicks', 'AvgLen')],
                    "crosses_opportunities": goalie_stat[('Crosses', 'Opp')],
                    "crosses_stops": goalie_stat[('Crosses', 'Stp')],
                    "crosses_stop_percentage": goalie_stat[('Crosses', 'Stp%')],
                    "opa": goalie_stat[('Sweeper', '#OPA')],
                    "opa_average_distance": goalie_stat[('Sweeper', 'AvgDist')],
                    "season_name": season_directory,
                })

    game_weeks_query = """
    UNWIND $game_weeks AS gw
    MERGE (s:Season {name: gw.season_name})
    MERGE (game_week:GameWeek {gw_number: gw.gw_number, season: gw.season_name})
    MERGE (game_week)-[:IS_A_GW]->(s)
    """

        
    player_creation_query = """
            UNWIND $players AS player
            MERGE (p:Player {name: player.Name})
            SET p.position = player.position,
                p.footed = player.footed,
                p.height = player.height,
                p.weight = player.weight,
                p.dob = player.dob,
                p.birth_place = player.birth_place,
                p.national_team = player.national_team,
                p.club = player.club,
                p.wages = player.wages,
                p.contract_expiry = player.contract_expiry,
                p.img = player.img

            MERGE (team:Team {name: player.Team})
            MERGE (squad: Squad{name: player.Team + " " + $season_name})
            MERGE (team)-[:HAS_SQUAD]->(squad)
            MERGE (p)-[:IS_IN_SQUAD]->(squad)
    """

    
    # UNWIND query
    match_creation_query = """
    UNWIND $matches AS match_data
    CREATE (match:Match {id: match_data.match_id})
    SET match.date = match_data.date,
        match.time = match_data.time
    MERGE (venue:Venue {name: match_data.venue})
    MERGE (match)-[:PLAYED_AT]->(venue)
    WITH match, match_data
    MATCH (home_team:Squad {name: match_data.home_team + " " + $season_name})
    MATCH (away_team:Squad {name: match_data.away_team + " " + $season_name})
    MATCH (game_week:GameWeek {gw_number: match_data.gameweek, season: $season_name})
    MERGE (match)-[:HOME_TEAM]->(home_team)
    MERGE (match)-[:AWAY_TEAM]->(away_team)
    MERGE (match)-[:IS_OF_GW]->(game_week)
    FOREACH (_ IN CASE WHEN match_data.has_match_report THEN [1] ELSE [] END |
        SET match.home_score = match_data.home_score,
            match.away_score = match_data.away_score
        FOREACH (_ IN CASE WHEN match_data.referee IS NOT NULL THEN [1] ELSE [] END |
            MERGE (referee:Referee {name: match_data.referee})
            MERGE (match)-[:HAS_REFEREE]->(referee)
        )
        FOREACH (_ IN CASE WHEN match_data.home_expected IS NOT NULL THEN [1] ELSE [] END |
            SET match.home_expected = match_data.home_expected,
                match.away_expected = match_data.away_expected
        )
    )
    """
    
    # Player Stats Query
    player_query = """
    UNWIND $player_stats AS ps
    MATCH (player:Player {name: ps.player_name})
    MATCH (squad:Squad {name: ps.team_name + " " + ps.season_name})
    MATCH (player)-[:IS_IN_SQUAD]->(squad)
    MATCH (match:Match {id: ps.match_id})
    MERGE (player)-[h:PLAYS_IN]->(match)
    SET h.goals = ps.goals,
        h.position = ps.position,
        h.assists = ps.assists,
        h.penalties = ps.penalties,
        h.shots = ps.shots,
        h.shots_on_target = ps.shots_on_target,
        h.touches = ps.touches,
        h.tackles = ps.tackles,
        h.interceptions = ps.interceptions,
        h.blocks = ps.blocks,
        h.xg = ps.xg,
        h.npxg = ps.npxg,
        h.xag = ps.xag,
        h.sca = ps.sca,
        h.gca = ps.gca,
        h.passes_completed = ps.passes_completed,
        h.passes_attempted = ps.passes_attempted,
        h.pass_accuracy = ps.pass_accuracy,
        h.progressive_passes = ps.progressive_passes,
        h.carries = ps.carries,
        h.progressive_carries = ps.progressive_carries;
    """
    
    # Goalie Stats Query
    goalie_query = """
    UNWIND $goalkeeper_stats AS gs
    MATCH (goalie:Player {name: gs.goalie_name})
    MATCH (squad:Squad {name: gs.team_name + " " + gs.season_name})
    MATCH (goalie)-[:IS_IN_SQUAD]->(squad)
    MATCH (match:Match {id: gs.match_id})
    MERGE (goalie)-[h:PLAYS_IN]->(match)
    SET h.minutes_played = gs.minutes_played,
        h.sota = gs.sota,
        h.goals_allowed = gs.goals_allowed,
        h.saves = gs.saves,
        h.save_percentage = gs.save_percentage,
        h.psxg = gs.psxg,
        h.passes_completed = gs.passes_completed,
        h.passes_attempted = gs.passes_attempted,
        h.pass_accuracy = gs.pass_accuracy,
        h.gk_passes_attempted = gs.gk_passes_attempted,
        h.throws = gs.throws,
        h.launch_percentage = gs.launch_percentage,
        h.launch_average_length = gs.launch_average_length,
        h.goal_kicks_attempted = gs.goal_kicks_attempted,
        h.goal_kick_launch_percentage = gs.goal_kick_launch_percentage,
        h.goal_kick_average_length = gs.goal_kick_average_length,
        h.crosses_opportunities = gs.crosses_opportunities,
        h.crosses_stops = gs.crosses_stops,
        h.crosses_stop_percentage = gs.crosses_stop_percentage,
        h.opa = gs.opa,
        h.opa_average_distance = gs.opa_average_distance;
    """
    
    # Execute Queries
    with driver.session(database="neo4j") as session:
        session.run(game_weeks_query, game_weeks=game_weeks_data)
        session.run(player_creation_query, players=player_details, season_name = season_directory)
        session.run(match_creation_query, matches=match_data, season_name=season_directory)
        session.run(player_query, player_stats=player_stats_data)
        session.run(goalie_query, goalkeeper_stats=goalkeeper_stats_data)
    
    # Close the Neo4j driver



2021-2022
the length of fixtures df is 380
2022-2023
the length of fixtures df is 380
2023-2024
the length of fixtures df is 380
2024-2025
the length of fixtures df is 380
